In [1]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import nltk
import string
import spacy
from tqdm import tqdm

nltk.download("wordnet")
nltk.download("omw-1.4")  

from nltk.corpus import wordnet as wn
from datasets.loader import load_splits, save_splits
from datasets.paths import ProjectPaths


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\malos\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\malos\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [2]:
paths = ProjectPaths()

In [3]:
pd.set_option("display.max_colwidth", None)


In [4]:
syns = wn.synsets("program")
print(syns[0].name())
print(syns[0].lemmas()[0].name())
print(syns[0].definition())
print(syns[0].examples())


plan.n.01
plan
a series of steps to be carried out or goals to be accomplished
['they drew up a six-step plan', 'they discussed plans for a new bond issue']


In [5]:
nlp = spacy.load("es_core_news_sm")
punct = set(string.punctuation)

pos_map = {
	"NOUN": "n",
	"VERB": "v",
	"ADJ": "a",
	"ADV": "r"
}

print(punct)


{'@', '\\', '.', '+', '(', ';', '=', '^', '?', '-', "'", '>', '$', ',', '/', '`', '%', ']', '_', '"', '[', '!', '|', ')', '#', '<', '*', '&', ':', '}', '~', '{'}


In [6]:
ALLOWED_CHARS = set(" abcdefghijklmnopqrstuvwxyzñáéíóú")

def normalize(text: str):
	return "".join(
		char for char in text.replace("_", " ").replace("-", " ").lower()
		if char in ALLOWED_CHARS
	)

def extract_synonyms(word: str, wn_pos=None) -> set:
	synonyms = set()
	synsets = wn.synsets(word, lang="spa", pos=wn_pos) if wn_pos else wn.synsets(word, lang="spa")
	
	for syn in synsets:
		for lemma in syn.lemmas(lang="spa"):
			synonyms.add(normalize(lemma.name()))
	
	return synonyms

def get_synonyms(word: str, pos: str):
	word = word.strip()
	word_lower = word.lower()
	
	wn_pos = pos_map.get(pos)
	
	synonyms = extract_synonyms(word_lower, wn_pos)
	synonyms.add(word_lower)
	
	# Si no se encuentran sinónimos, a excepción del original, se intenta eliminar la puntuación
	if len(synonyms) <= 1:
		word_clean = "".join(char for char in word if char not in punct).lower()
		if word_clean != word_lower:
			synonyms.update(extract_synonyms(word_clean, wn_pos))
			synonyms.add(word_clean)

	# Construir distribución de probabilidad: la palabra original obtiene 0.5 y las demás comparten el resto
	n = len(synonyms)
	if n <= 1:
		probabilities = [1.0]
	else:
		probabilities = [0.5 if w == word_lower else 0.5 / (n - 1) for w in synonyms]

	return list(synonyms), probabilities

In [7]:
get_synonyms("hombre", "NOUN")


(['varón', 'hombre', 'mundo', 'esposo', 'humanidad', 'marido'],
 [0.1, 0.5, 0.1, 0.1, 0.1, 0.1])

In [8]:
def match_case(original: str, new: str):
	if original.isupper():
		return new.upper()
	if original[0].isupper():
		return new.capitalize()
	return new


In [9]:
def augment_sentence(sentence: str):
	doc = nlp(sentence)
	orig_text = sentence

	tokens_info = []
	for i, token in enumerate(doc):
		end = token.idx + len(token.text)
		if i < len(doc) - 1:
			next_start = doc[i+1].idx
			# print(end, next_start)
			whitespace = orig_text[end:next_start]
		else:
			whitespace = ""
		tokens_info.append({
			"token": token,
			"whitespace": whitespace
		})

	new_tokens = []
	for info in tokens_info:
		token = info["token"]
		token_text = token.text
		if token.is_punct or token.is_space or token.is_digit or token.is_stop:
			new_tokens.append(token_text)
		else:
			pos = token.pos_
			syns, probs = get_synonyms(token_text, pos)
			chosen = np.random.choice(syns, p=probs)
			chosen = match_case(token.text, chosen.strip())
			new_tokens.append(chosen)

	parts = []
	for info, new_tok in zip(tokens_info, new_tokens):
		parts.append(new_tok + info["whitespace"])
	return "".join(parts)


In [10]:
sentence = "Un  \nhombre."
augment_sentence(sentence)


'Un  \nhombre.'

In [11]:
def augment_row(row: pd.Series, n_augmentations: int):
	sentence1 = row["sentence1"]
	sentence2 = row["sentence2"]
	
	sentence1_variants = [sentence1]
	sentence2_variants = [sentence2]

	for _ in range(n_augmentations):
		sentence1_variants.append(augment_sentence(sentence1))
		sentence2_variants.append(augment_sentence(sentence2))

	# all_pairs = list(product(sentence1_variants, sentence2_variants))
		
	seen = set()
	unique_pairs = []
	for sentence1, sentence2 in zip(sentence1_variants, sentence2_variants):
		pair_key = tuple(sorted([sentence1, sentence2]))
		if pair_key not in seen:
			seen.add(pair_key)
			unique_pairs.append({
				"sentence1": sentence1,
				"sentence2": sentence2,
				"score": row["score"],
				"split": row["split"],
				"score_norm": row["score_norm"]
			})

	new_df = pd.DataFrame(unique_pairs)
	n_new_unique = len(new_df) - 1

	return new_df, n_new_unique


In [12]:
splits = {
	"train": paths.processed_dir
}

dfs = load_splits(splits)
df = dfs["train"]

print(len(df))

5741


In [13]:
augmented_dfs = []
failed_rows = []

total_rows = len(df)
n_augmentations = 2
total_new_examples = 0

for idx, row in tqdm(df.iterrows(), total=total_rows, desc="Augmenting data"):
	new_df, n_new_unique = augment_row(row, n_augmentations=n_augmentations)

	augmented_dfs.append(new_df)
	total_new_examples += n_new_unique

	if n_new_unique <= 0:
		failed_rows.append({
			"index": idx,
			"sentence1": row["sentence1"],
			"sentence2": row["sentence2"]
		})

augmented_df = pd.concat(augmented_dfs, ignore_index=True)

before_global = len(augmented_df)
augmented_df = augmented_df.drop_duplicates(subset=["sentence1", "sentence2"])
after_global = len(augmented_df)

efficiency = total_new_examples / (total_rows * n_augmentations)

print(f"\nAugmentation Summary:")
print(f"- Original dataset size: {total_rows}")
print(f"- Total rows after augmentation (before global dedup): {before_global}")
print(f"- Final dataset size (after global dedup): {after_global}")
print(f"- Total new unique examples added: {after_global - total_rows}")
print(f"- Augmentation efficiency: {efficiency:.2%}")

print(f"\nRows without successful augmentations: {len(failed_rows)}")
if failed_rows:
	failed_df = pd.DataFrame(failed_rows)
	display(failed_df)


Augmenting data: 100%|██████████| 5741/5741 [03:22<00:00, 28.33it/s]



Augmentation Summary:
- Original dataset size: 5741
- Total rows after augmentation (before global dedup): 15257
- Final dataset size (after global dedup): 15206
- Total new unique examples added: 9465
- Augmentation efficiency: 82.88%

Rows without successful augmentations: 433


,index,sentence1,sentence2
0,3,Tres hombres están jugando al ajedrez.,Dos hombres están jugando al ajedrez.
1,5,Algunos hombres están luchando.,Dos hombres están luchando.
2,16,El oso polar se está deslizando en la nieve.,Un oso polar se está deslizando por la nieve.
3,21,Un hombre está tocando una guitarra.,Una chica está tocando una guitarra.
4,22,Un panda se está deslizando por un tobogán.,Un panda se desliza por un tobogán.
...,...,...,...
428,5680,El príncipe William se viste de samurái en su gira por Japón,El Príncipe Guillermo de Gran Bretaña llega a Beijing
429,5685,Texas demanda a los refugiados sirios,"Turquía ""explota"" a los refugiados sirios"
430,5687,5 muertos en ataques aéreos israelíes en Gaza,"38 militantes del IS muertos en enfrentamientos, ataques aéreos en Irak"
431,5701,Miles cruzan la frontera entre Austria y Hungría,Miles de rusos varados en el extranjero


In [14]:
display(augmented_df.head(20))


,sentence1,sentence2,score,split,score_norm
0,Un avión está despegando.,Un avión está despegando.,5.00,train,1.00
1,Un avión está despegando.,Un aeroplano está despegando.,5.00,train,1.00
2,Un aeroplano está despegando.,Un aeroplano está despegando.,5.00,train,1.00
3,Un hombre está tocando una gran flauta.,Un hombre está tocando una flauta.,3.80,train,0.76
4,Un esposo está tocando una gran flauta.,Un hombre está tocando una flauta.,3.80,train,0.76
5,Un hombre está untando queso rallado en una pizza.,Un hombre está untando queso rallado en una pizza cruda.,3.80,train,0.76
6,Un marido está untando queso rallado en una pizza.,Un humanidad está untando queso rallado en una pizza cruda.,3.80,train,0.76
7,Un marido está untando queso rallado en una pizza.,Un esposo está untando queso rallado en una pizza cruda.,3.80,train,0.76
8,Tres hombres están jugando al ajedrez.,Dos hombres están jugando al ajedrez.,2.60,train,0.52
9,Un hombre está tocando el violonchelo.,Un hombre sentado está tocando el violonchelo.,4.25,train,0.85


In [15]:
display(augmented_df.tail(20))


,sentence1,sentence2,score,split,score_norm
15237,"Francia cierra la mezquita, arresta a un hombre en la represión después de los ataques",La seguridad se reforzó en las iglesias de Nueva Delhi después de los ataques,2.0,train,0.4
15238,"Francia cierra la mezquita, arresta a un esposo en la supresión después de los ataques",La aplomo se reforzó en las iglesias de Nueva Delhi después de los ataques,2.0,train,0.4
15239,"Francia cierra la mezquita, arresta a un humanidad en la represión después de los ataques",La departamento de seguridad se reforzó en las iglesias de Nueva Delhi después de los ataques,2.0,train,0.4
15240,Se informa que un avión ruso se ha estrellado en Egipto,Piloto muerto al estrellarse un avión de EE.UU. en Inglaterra,2.0,train,0.4
15241,Se informa que un avión ruso se ha estrellado en Egipto,Aeronauta muerto al estrellarse un aeroplano de EE.UU. en Inglaterra,2.0,train,0.4
15242,Se informa que un avión ruso se ha estrellado en Egipto,Aviador muerto al estrellar un avión de ESTADOS UNIDOS en Inglaterra,2.0,train,0.4
15243,Vendavales severos mientras la tormenta Clodagh golpea a Gran Bretaña,Merkel promete la solidaridad de la OTAN con Letonia,0.0,train,0.0
15244,Vendavales severos mientras la tormenta Clodagh golpea a Gran Bretaña,Merkel promete la compañerismo de la ORGANIZACIÓN DEL TRATADO DEL ATLÁNTICO NORTE con Letonia,0.0,train,0.0
15245,Vendavales severos mientras la tormenta violenta Clodagh golpea a Gran Bretaña,Merkel promete la solidaridad de la OTAN con Letonia,0.0,train,0.0
15246,Docenas de egipcios rehenes tomados por terroristas libios como venganza por los ataques aéreos,El número de muertos en el accidente de un barco egipcio aumenta a medida que se encuentran más cuerpos en el Nilo,0.0,train,0.0


In [16]:
save_splits(augmented_df, paths.augmented_dir)
